# Functions with example of usage to prepare datesets for MOFA model training

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
def load_all_datasets_from_folder(folderpath, sep=";", encoding="utf-8", header=0, index_col=None):
    """
    Load all datasets (CSV FORMAT !!!) from a folder into a dictionary of pandas dataframes.

    Parameters
    ----------
    folderpath : str
        The path to the folder containing the datasets.
    sep : str or list, optional
        The separator used in the CSV files. The default is ";".
    encoding : str or list, optional
        The encoding used in the CSV files. The default is "utf-8".
    header : int or list, optional
        The row number to use as the column names. The default is 0.
    index_col : int or list, optional
        The column to use as the index. The default is None.
    """
    datasets = {}
    csv_files = [f for f in os.listdir(folderpath) if f.endswith(".csv")]
    
    if not isinstance(sep, list):
        sep = [sep] * len(csv_files)
    else:
        if len(sep) != len(csv_files):
            raise ValueError("The length of the list 'sep' must be equal to the number of datasets in the folder")
    
    if not isinstance(encoding, list):
        encoding = [encoding] * len(csv_files)
    else:
        if len(encoding) != len(csv_files):
            raise ValueError("The length of the list 'encoding' must be equal to the number of datasets in the folder")
    
    if not isinstance(header, list):
        header = [header] * len(csv_files)
    else:
        if len(header) != len(csv_files):
            raise ValueError("The length of the list 'header' must be equal to the number of datasets in the folder")
    
    if not isinstance(index_col, list):
        index_col = [index_col] * len(csv_files)
    else:
        if len(index_col) != len(csv_files):
            raise ValueError("The length of the list 'index_col' must be equal to the number of datasets in the folder")
    
    for k, filename in enumerate(csv_files):
        name = filename.split(".")[0]
        datasets[name] = pd.read_csv(os.path.join(folderpath, filename), sep=sep[k], encoding=encoding[k], header=header[k], index_col=index_col[k])
    
    return datasets

In [ ]:
datasets = load_all_datasets_from_folder("Multiomics_ENIGMA_small_v4", sep=[',', ';', ';', ';'], encoding="utf-8", header=0, index_col=None)
datasets.keys()

In [ ]:
def PySPRESSO_to_MOFA(dataset, group='lipidomics', view='POS', feature_column='cpdID', sample_column_prefix="Area: ", sample_column_end_char = '_'):
    """
    Convert a PySPRESSO eligible (lipidomics, metabolomics) dataset to a MOFA+ compatible format.

    Parameters
    ----------
    dataset : pandas.DataFrame
        The dataset to convert
    omic : str
        The type of omic data (e.g. lipidomics, metabolomics); is included in the group column in MOFA+ format
    view : str
        The view of the omic data (e.g. positive, negative); is included in the view column (second column to distinguish analyses) in MOFA+ format
    feature_column : str
        The name of the column containing the feature IDs; is included in the feature column in MOFA+ format
    sample_column_prefix : str
        The prefix of the columns containing (the sample names) and values; is included in the sample column and values in value column in MOFA+ format
    ----------
    Returns MOFA+ compatible data:
        pandas.DataFrame with the following columns:
        - sample (str): the sample name
        - group (str): the group (omic type) of the sample (e.g. lipidomics, metabolomics)
        - feature (str): the feature ID
        - view (str): the view of the omic data (e.g. positive, negative)
        - value (float): the value of the feature in the sample
    """    
    # Extract the sample names (columns named Area: SAMPLE_NAME)
    sample_columns = dataset.columns[dataset.columns.str.startswith(sample_column_prefix)]
    samples = sample_columns.str.replace(sample_column_prefix, "")
    # Keep only string until the first occurence of sample_column_end_char (typically '_')
    samples = samples.str.split(sample_column_end_char).str[0]

    # Extract the feature names (column named IMTM_Feature_ID)
    features = dataset[feature_column]

    # Extract the values
    values = dataset[sample_columns].values

    # Create group and view columns
    group = np.repeat(group, len(samples) * len(features))

    # Create the dataframe with the MOFA+ format
    data = pd.DataFrame({
        "sample": np.repeat(samples, len(features)),
        "group": group,
        "feature": np.tile(features, len(samples)),
        "view": np.repeat(view, len(samples) * len(features)),
        "value": values.flatten()
    })
    return data

def Proteomics_to_MOFA(dataset, group='proteomics', view='proteomics', feature_column='Description', sample_column_prefix='SampleID:'):
    """
    Convert a proteomics dataset to a MOFA+ compatible format.

    Parameters
    ----------
    dataset : pandas.DataFrame
        The dataset to convert
    omic : str
        The type of omic data (e.g. proteomics); is included in the group column in MOFA+ format
    view : str
        The view of the omic data (e.g. proteomics); is included in the view column (second column to distinguish analyses) in MOFA+ format
    feature_column : str
        The name of the column containing the feature IDs; is included in the feature column in MOFA+ format
    sample_column_prefix : str
        The prefix of the columns containing the sample names; is included in the sample column in MOFA+ format
    ----------
    Returns MOFA+ compatible data:
        pandas.DataFrame with the following columns:
        - sample (str): the sample name
        - group (str): the group (omic type) of the sample (e.g. proteomics)
        - feature (str): the feature ID
        - view (str): the view of the omic data (e.g. proteomics)
        - value (float): the value of the feature in the sample
    """
    # Extract the sample names (columns named SampleID: SAMPLE_NAME)
    sample_columns = dataset.columns[dataset.columns.str.startswith(sample_column_prefix)]
    samples = sample_columns.str.replace(sample_column_prefix, "")

    # Extract the feature names (column named Description)
    features = dataset[feature_column]

    # Extract the values
    values = dataset[sample_columns].values

    # Create group and view columns
    group = np.repeat(group, len(samples) * len(features))

    # Create the dataframe with the MOFA+ format
    data = pd.DataFrame({
        "sample": np.repeat(samples, len(features)),
        "group": group,
        "feature": np.tile(features, len(samples)),
        "view": np.repeat(view, len(samples) * len(features)),
        "value": values.flatten()
    })

    return data


def Raman_to_MOFA(dataset, group='raman', view = 'spectroscopy', feature_column_prefix = 'X', sample_ID='ID', sample_column_end_char = '_'):
    """
    Convert a Raman dataset to a MOFA+ compatible format.

    Parameters
    ----------
    dataset : pandas.DataFrame
        The dataset to convert
    omic : str
        The type of omic data (e.g. raman); is included in the group column in MOFA+ format
    view : str
        The view of the omic data (e.g. spectroscopy); is included in the view column (second column to distinguish analyses) in MOFA+ format
    metadata_columns_lst : list
        The list of metadata columns to include in the metadata dataframe; default is ['Sex', 'Age', 'BMI']
    feature_column_prefix : str
        The prefix of the columns containing the feature values; is included in the feature column in MOFA+ format; default is 'X'
    sample_ID : str
        The name of the column containing the sample IDs; is included in the sample column in MOFA+ format; default is 'ID'
    sample_column_end_char : str
        The character separating the sample ID from the rest of the column name; default is '_'
    """

    
    # Extract values for the features
    features_columns_lst = [col for col in dataset.columns if col.startswith(feature_column_prefix)]
    values = dataset[features_columns_lst].values

    # Create sample, omic and view columns
    samples = np.repeat(dataset[sample_ID], len(features_columns_lst))
    # Keep only string until the first occurence of sample_column_end_char (typically '_')
    samples = samples.str.split(sample_column_end_char).str[0]

    group = np.repeat(group, len(dataset) * len(features_columns_lst))
    view = np.repeat(view, len(dataset) * len(features_columns_lst))

    # Create the dataframe with the MOFA+ format
    data = pd.DataFrame({
        "sample": samples,
        "group": group,
        "feature": np.tile(features_columns_lst, len(dataset)),
        "view": view,
        "value": values.flatten()
    })

    return data




In [ ]:
def handle_non_numeric_values(data, dict):
    """
    Handle non-numeric values in the MOFA+ data. 

    Parameters
    ----------
    data : pandas.DataFrame
        The MOFA+ data to handle
    dict : dict
        The dictionary containing the non-numeric values to replace with the corresponding numeric values. e.g. {'Female': 0, 'Male': 1, ...}
    """
    # Find all features with non-numeric values
    non_numeric = data[~data['value'].apply(lambda x: str(x).replace('.', '').replace('-', '').replace('e', '').isdigit())]
    non_numeric['value'].unique()

    # Replace the non-numeric values with the values from the dictionary
    data['value'] = data['value'].replace(dict)
    data = data.astype({'value': float})
    return data 

In [ ]:
list_of_keys = list(datasets.keys())

In [ ]:
MOFA_datasets = {}

### Example:

In [ ]:
data = PySPRESSO_to_MOFA(datasets[list_of_keys[2]], group = 'ENIGMA_small', view = 'lipidomics' , feature_column = 'IMTM_Feature_ID', sample_column_prefix = "Area: ", sample_column_end_char='_NEG')
# Replace 'L' at the start of sample names with 'ID'
data['sample'] = data['sample'].str.replace('L', 'ID')
# Drop Blanks and SRM and QC samples
data = data[~data['sample'].str.contains("Blank")]
data = data[~data['sample'].str.contains("SRM")]
data = data[~data['sample'].str.contains("QC")]
print(data['sample'].unique())
print(len(data['sample'].unique()))
# Add NG/POS to the feature names
data['feature'] = data['feature'] + '_NEG'
MOFA_datasets[list_of_keys[2]] = data
data

In [ ]:
def merge_MOFA_datasets(MOFA_datasets_dict):
    """
    Merge multiple MOFA+ compatible datasets into a single MOFA+ compatible dataset.

    Parameters
    ----------
    MOFA_datasets_dict : dict
        A dictionary containing the MOFA+ compatible datasets to merge
    """
    return pd.concat(MOFA_datasets_dict.values(), ignore_index=True)

In [ ]:
merged_data = merge_MOFA_datasets(MOFA_datasets)
merged_data

In [ ]:
# Save the merged data to a CSV file
name = "ENIGMA_small_v2-4"
merged_data.to_csv(f"{name}_merged_MOFA_data.csv", index=False, sep=";", encoding="utf-8")